# Libro de órdenes con Conv1D — ¿hay señal solo en el libro?

Hasta aquí veníamos usando *resúmenes* del libro (mid, spread, microprice, imbalance).
La pregunta de este cuaderno es más ambiciosa y más limpia: si nos quedamos **solo con
el libro de órdenes visible** de Polymarket, sin atajos ni features externas, ¿queda
señal predictiva dentro?

Es un experimento de *representación*: queremos saber si el libro, tratado como un
objeto con estructura (niveles × tiempo) y no como una tabla plana, contiene
información sobre hacia dónde se moverá el precio. La arquitectura está inspirada en
DeepLOB (convolución sobre el libro + recurrencia para la dinámica temporal).

> Aviso importante: aquí **no se reentrena nada pesado**. El entrenamiento vive en
> `scripts/experiments/book_only_conv1d_baseline_v1.py`. Este cuaderno carga los
> resultados ya generados y los explica.


## Qué estamos probando

El libro se convierte en un **tensor** de forma `(secuencia, paso, nivel, canal)`:

- **secuencia**: cada ejemplo es una ventana de varios instantes consecutivos;
- **paso**: cada foto del libro dentro de esa ventana (la malla es de 2 s);
- **nivel**: cada escalón de precio del libro (usamos los primeros niveles, los que
  de verdad tienen liquidez);
- **canal**: lo que guardamos de cada nivel — tamaño, acumulado, distancia al mid y
  una máscara de presencia (si ese nivel existe o está vacío).

Comparamos dos modelos, de menos a más complejo:

- **`book_last_snapshot_conv`**: mira **solo la última foto** antes de decidir. Es el
  control: ¿basta con la instantánea actual del libro?
- **`book_seq_conv_gru`**: resume cada foto con una **Conv1D** (el *BookEncoder*: capta
  desbalances y gradientes de profundidad entre niveles) y luego pasa la secuencia de
  resúmenes por una **GRU**, que modela la dinámica temporal. Es la versión que sí
  aprovecha el orden en el tiempo.

La idea es ver si añadir la dimensión temporal (GRU) aporta algo sobre la foto suelta.


In [ ]:
from pathlib import Path
import json
import pandas as pd
OUT = Path('../data/experiments/book_only_conv1d_baseline_v1')
decision = json.loads((OUT / 'decision.json').read_text(encoding='utf-8'))
metrics = pd.read_csv(OUT / 'model_metrics.csv')
selected = pd.read_csv(OUT / 'selected_candidates.csv')
days = pd.read_csv(OUT / 'day_breakdown.csv')
decision


## Métricas de ranking

Aquí no miramos *accuracy* (recuérdese que la clase FLAT domina y la haría engañosa).
Miramos la capacidad de **ordenar**: si el modelo puntúa alto los instantes que de
verdad se mueven a favor. Las columnas habituales son el **AUC** (capacidad de
separar, 0,5 = azar) y la **correlación de rango** entre la puntuación del modelo y el
movimiento real. Son métricas de *representación*: dicen si el modelo "ve" algo, no si
ese algo se traduce en una política rentable.


In [ ]:
metrics.round(5)


## Candidatos seleccionados por validación

Los umbrales y el candidato se eligen **solo con validación**, nunca mirando el test
terminal. Esta tabla recoge las configuraciones que sobreviven a ese filtro. Que un
candidato aparezca aquí significa que ordena bien en datos que no vio al entrenar; aún
no significa que gane dinero tras costes.


In [ ]:
selected.head(20).round(5)


## Desglose diario del mejor candidato

El promedio global puede esconder que la señal vive en uno o dos días buenos y se cae
el resto. Por eso miramos el resultado **día a día**: una política utilizable tiene que
ser razonablemente estable, no depender de un único día afortunado.


In [ ]:
best = decision.get('best_policy', {})
mask = days['model_name'].eq(best.get('model_name')) & days['policy_name'].eq(best.get('policy_name'))
days[mask].sort_values(['terminal_split','session_day']).round(5)


## Lectura y decisión

El patrón que se repite en todo el bloque de aprendizaje profundo aparece también aquí:
el modelo **aprende representación** (la versión con Conv1D + GRU ordena mejor que la
foto suelta, señal de que la dinámica temporal del libro aporta), pero esa ventaja **no
se sostiene como política** cuando se exige selección causal estricta y estabilidad
día a día. Con ~15 días de datos no hay suficiente para que una red convierta esa
representación en una regla de operación fiable.

Conclusión honesta: el libro contiene señal, pero el deep learning, con estos datos, no
la convierte en una política mejor que el especialista tabular. Se documenta como paso
necesario del arco —probar que la complejidad no se justifica todavía— y se aparca a la
espera de muchos más días. Ver `docs/BOOK_ONLY_CONV1D_BASELINE_V1.md`.
